In [1]:
%load_ext autoreload
%autoreload 2

In [10]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from shoreshop3_analysis import data

In [3]:
basedir = Path.home() / 'abmurraylab/aidan_output'

diffusion_1984_2024_path = basedir / 'mipNC_1984-2024_DealyDiffusion.csv'
gradient_1984_2024_path = basedir / 'mipNC_1984-2024_DealyGradient.csv'
diffusion_2020_2100_path = basedir / 'mipNC_2020-2100_DealyDiffusion.csv'
gradient_2020_2100_path = basedir / 'mipNC_2020-2100_DealyGradient.csv'

In [4]:
transects = xr.open_dataset(data.nc_1980_2023_cube_path).transect.values.tolist()
len(transects)

970

In [26]:
keep = ['Unnamed: 0'] + transects

diffusion_1984_2024 = pd.read_csv(diffusion_1984_2024_path, usecols=keep)
gradient_1984_2024 = pd.read_csv(gradient_1984_2024_path, usecols=keep)
diffusion_2020_2100 = pd.read_csv(diffusion_2020_2100_path, usecols=keep)
gradient_2020_2100 = pd.read_csv(gradient_2020_2100_path, usecols=keep)

for df in [diffusion_1984_2024, gradient_1984_2024, diffusion_2020_2100, gradient_2020_2100]:
    df.rename(columns={'Unnamed: 0': 'time'}, inplace=True)
    df['time'] = pd.to_datetime(df['time'])

In [6]:
diffusion_1984_2024

,time,usa_NC_0001_0001,usa_NC_0001_0010,usa_NC_0001_0020,usa_NC_0001_0030,usa_NC_0001_0040,usa_NC_0001_0050,usa_NC_0001_0060,usa_NC_0001_0070,usa_NC_0001_0080,...,usa_NC_0049_0141,usa_NC_0049_0151,usa_NC_0049_0161,usa_NC_0049_0171,usa_NC_0049_0181,usa_NC_0049_0191,usa_NC_0049_0201,usa_NC_0049_0211,usa_NC_0049_0221,usa_NC_0049_0229
0,1984-01-01,71.4,108.7,76.2,56.2,40.5,40.6,33.8,31.6,37.0,...,28.8,28.7,40.2,34.2,16.9,10.1,17.5,10.3,25.1,49.1
1,1984-01-02,71.4,108.7,76.2,56.3,40.5,40.6,33.9,31.7,37.0,...,28.8,28.7,40.2,34.2,16.9,10.1,17.5,10.3,25.1,49.1
2,1984-01-03,71.4,108.7,76.2,56.3,40.5,40.6,33.9,31.7,37.0,...,28.8,28.7,40.2,34.2,16.9,10.1,17.5,10.3,25.1,49.1
3,1984-01-04,71.4,108.7,76.2,56.3,40.5,40.6,33.9,31.7,37.0,...,28.8,28.7,40.2,34.2,16.9,10.1,17.5,10.3,25.1,49.1
4,1984-01-05,71.4,108.7,76.2,56.3,40.5,40.6,33.9,31.7,37.0,...,28.8,28.7,40.2,34.2,16.9,10.1,17.5,10.3,25.1,49.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14567,2023-11-19,15.7,61.0,98.8,115.0,75.0,112.4,53.3,67.7,97.1,...,-56.0,2.8,51.6,51.2,-6.9,-56.3,15.6,39.8,-49.8,29.9
14568,2023-11-20,15.7,61.0,98.8,115.0,75.0,112.4,53.3,67.7,97.1,...,-56.0,2.8,51.6,51.2,-6.9,-56.3,15.6,39.8,-49.8,29.9
14569,2023-11-21,15.7,61.0,98.8,115.0,75.0,112.4,53.3,67.7,97.1,...,-56.0,2.7,51.6,51.2,-6.9,-56.3,15.6,39.8,-49.8,29.9
14570,2023-11-22,15.7,61.0,98.8,115.0,75.0,112.4,53.3,67.7,97.1,...,-56.0,2.7,51.6,51.2,-6.9,-56.3,15.6,39.8,-49.8,29.9


In [27]:
def fix(df, model):
    df['model'] = model
    return df.melt(id_vars=['model', 'time'], var_name='transect', value_name='shoreline_position')

In [8]:
data_1984_2024 = pd.concat([
    fix(diffusion_1984_2024, 'DealyDiffusion'),
    fix(gradient_1984_2024, 'DealyGradient'),
]).set_index(['model', 'transect', 'time']).to_xarray()

data_1984_2024

/tmp/ipykernel_208663/3503649767.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_208663/3503649767.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


<xarray.Dataset> Size: 226MB
Dimensions:             (model: 2, transect: 970, time: 14572)
Coordinates:
  * model               (model) object 16B 'DealyDiffusion' 'DealyGradient'
  * transect            (transect) object 8kB 'usa_NC_0001_0001' ... 'usa_NC_...
  * time                (time) datetime64[us] 117kB 1984-01-01 ... 2023-11-23
Data variables:
    shoreline_position  (model, transect, time) float64 226MB 71.4 71.4 ... 44.0

In [28]:
data_2020_2100 = pd.concat([
    fix(diffusion_2020_2100, 'DealyDiffusion'),
    fix(gradient_2020_2100, 'DealyGradient'),
]).set_index(['model', 'transect', 'time']).to_xarray()

data_2020_2100

/tmp/ipykernel_208663/1957292450.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_208663/1957292450.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


<xarray.Dataset> Size: 459MB
Dimensions:             (model: 2, transect: 970, time: 29587)
Coordinates:
  * model               (model) object 16B 'DealyDiffusion' 'DealyGradient'
  * transect            (transect) object 8kB 'usa_NC_0001_0001' ... 'usa_NC_...
  * time                (time) datetime64[us] 237kB 2019-01-01 ... 2100-01-02
Data variables:
    shoreline_position  (model, transect, time) float64 459MB 24.3 24.3 ... 38.8

In [13]:
data_1984_2024_dir = data.basedir / 'aidan-1984-2024'
os.makedirs(data_1984_2024_dir, exist_ok=True)
data_1984_2024.to_netcdf(data_1984_2024_dir / 'models.nc', format='NETCDF4')

In [30]:
data_2020_2100_dir = data.basedir / 'aidan-2020-2100'
os.makedirs(data_2020_2100_dir, exist_ok=True)
data_2020_2100.to_netcdf(data_2020_2100_dir / 'models.nc', format='NETCDF4')